# Imports

In [ ]:
import os
import numpy as np
import re
import pandas as pd
import gc
from datetime import datetime, timedelta
from google.colab import drive

#Setup and Globals

In [ ]:
drive.mount('/content/drive', force_remount=True)
base_path = "/content/drive/MyDrive/EAI_Shared"                                 #qui cambiare in base al percorso della cartella di lavoro!
dataset = f"{base_path}/Data"
# Input path
db_18ch = f"{dataset}/03_CHB_MIT_WPE_Features_full_18ch"
summ_dir = f"{dataset}/summaries"
# Output path
db_final_std_v2 = f"{dataset}/04_CHB_MIT_WPE_Features_final_std"
os.makedirs(db_final_std_v2, exist_ok=True)

In [ ]:
# standard channels definition
channels = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]

# parameters configuration
CONFIG = {
    'WPE_DIR': db_18ch,
    'SUMMARIES_DIR': summ_dir,
    'EPOCH_DUR': 5.0,
    'STEP_DUR': 2.5,
    'PRE_ICTAL_WIN': 30 * 60,   # 30 minutes
    'SPH_WIN': 5 * 60,          # 5 minutes - Seizure Prediction Horizon
    'POST_ICTAL_GAP': 4 * 3600  # 4 hours - Post Critic Gap
}

# Run

In [ ]:
#support functions+parsing
def parse_time(time_str):
    return datetime.strptime(time_str.strip(), '%H:%M:%S')

def get_seconds_difference(start_str, end_str):
    try:
        t_start = parse_time(start_str)
        t_end = parse_time(end_str)
        if t_end <= t_start: t_end += timedelta(days=1)
        return (t_end - t_start).total_seconds()
    except: return 3600

def parse_comprehensive_summary(summary_path):
    files_metadata = []
    if not os.path.exists(summary_path): return []
    with open(summary_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    blocks = content.split("File Name: ")
    for block in blocks[1:]:
        data = {}
        lines = block.split("\n")
        data['filename'] = lines[0].strip()
        start_t = re.search(r"File Start Time: (\d{1,2}:\d{2}:\d{2})", block)
        end_t = re.search(r"File End Time: (\d{1,2}:\d{2}:\d{2})", block)
        duration_sec = get_seconds_difference(start_t.group(1), end_t.group(1)) if start_t and end_t else 3600
        data['duration_sec'] = int(duration_sec)
        data['n_epochs'] = int((duration_sec - CONFIG['EPOCH_DUR']) // CONFIG['STEP_DUR'] + 1)
        seizure_starts = re.findall(r"Seizure \d* ?Start Time: (\d+) seconds", block)
        seizure_ends = re.findall(r"Seizure \d* ?End Time: (\d+) seconds", block)
        data['seizures'] = list(zip([int(s) for s in seizure_starts], [int(e) for e in seizure_ends]))
        files_metadata.append(data)
    return files_metadata

In [ ]:
def label_patient(patient_id):
    """
    Labeling function
    """
    print(f"Elaborating patient {patient_id}")
    x_path = os.path.join(CONFIG['WPE_DIR'], f"{patient_id}_X_18ch.npy")
    if not os.path.exists(x_path): return

    X_full = np.load(x_path)
    total_rows = X_full.shape[0]
    summary_path = os.path.join(CONFIG['SUMMARIES_DIR'], f"{patient_id}-summary.txt")
    files_meta = parse_comprehensive_summary(summary_path)
    all_labels = np.zeros(total_rows, dtype=int)
    all_mask = np.ones(total_rows, dtype=bool)
    current_global_row = 0
    for f_meta in files_meta:
        file_offset_row = current_global_row
        n_epochs = min(f_meta['n_epochs'], total_rows - current_global_row)
        if n_epochs <= 0: break

        for (s_start, s_end) in f_meta['seizures']:
            #Pre-ictal definition
            pre_end_sec = s_start - CONFIG['SPH_WIN']
            pre_start_sec = max(0, pre_end_sec - CONFIG['PRE_ICTAL_WIN'])
            idx_pre_start = int(pre_start_sec // CONFIG['STEP_DUR'])
            idx_pre_end = int(pre_end_sec // CONFIG['STEP_DUR'])
            all_labels[file_offset_row + idx_pre_start : file_offset_row + idx_pre_end] = 1

            # Area to be discarded (SPH + crisis + 4h Post-Ictal) - also if pre-ictal falls here we discard it
            drop_start_sec = s_start - CONFIG['SPH_WIN']
            drop_end_sec = s_end + CONFIG['POST_ICTAL_GAP']
            idx_drop_start = int(max(0, drop_start_sec // CONFIG['STEP_DUR']))
            idx_drop_end = int(drop_end_sec // CONFIG['STEP_DUR'])
            actual_drop_start = file_offset_row + idx_drop_start
            actual_drop_end = min(total_rows, file_offset_row + idx_drop_end)
            all_mask[actual_drop_start : actual_drop_end] = False
        current_global_row += n_epochs

    indices_to_keep = np.where(all_mask)[0]
    if len(indices_to_keep) > 0:
        X_final = X_full[indices_to_keep]
        y_final = all_labels[indices_to_keep]
        np.save(os.path.join(CONFIG['WPE_DIR'], f"{patient_id}_X_labeled.npy"), X_final)
        np.save(os.path.join(CONFIG['WPE_DIR'], f"{patient_id}_y_labeled.npy"), y_final)
        print(f" Saved {len(y_final)} epochs (Pre-ictal: {np.sum(y_final)})")
    else:
        print(f"No valid data fot: {patient_id}")

In [ ]:
def create_robust_standardized_dataset(wpe_dir, output_path, baseline_minutes=20):
    """
    Standardization
    """
    all_x_std, all_y, all_groups = [], [], []
    n_baseline_epochs = int((baseline_minutes * 60) / CONFIG['STEP_DUR'])
    files = sorted([f for f in os.listdir(wpe_dir) if f.endswith('_X_labeled.npy')])
    print(f"Standardization on {len(files)} patients")
    for f in files:
        pid_str = f.replace('_X_labeled.npy', '')
        pid_int = int(''.join(filter(str.isdigit, pid_str)))
        X_p = np.load(os.path.join(wpe_dir, f)).astype(np.float32)
        y_p = np.load(os.path.join(wpe_dir, f"{pid_str}_y_labeled.npy"))
        interictal_indices = np.where(y_p == 0)[0]
        if len(interictal_indices) == 0: continue
        if len(interictal_indices) >= n_baseline_epochs:
            baseline_indices = interictal_indices[:n_baseline_epochs]
        else:
            baseline_indices = interictal_indices
        # Robust scaling on pure baseline
        X_baseline = X_p[baseline_indices]
        median_ref = np.median(X_baseline, axis=0)
        q1 = np.percentile(X_baseline, 25, axis=0)
        q3 = np.percentile(X_baseline, 75, axis=0)
        iqr_ref = q3 - q1 + 1e-6
        # clipping and transformation
        X_p_std = (X_p - median_ref) / iqr_ref
        X_p_std = np.clip(X_p_std, -5, 5)
        all_x_std.append(X_p_std)
        all_y.append(y_p)
        all_groups.append(np.full(len(y_p), pid_int))
        print(f"{pid_str} scaled - Baseline: {len(baseline_indices)} ephocs")

    #Final save
    np.save(os.path.join(output_path, "X_baseline_std.npy"), np.concatenate(all_x_std))
    np.save(os.path.join(output_path, "y_baseline_std.npy"), np.concatenate(all_y))
    np.save(os.path.join(output_path, "groups_baseline_std.npy"), np.concatenate(all_groups))
    print(f"DB saved in: {output_path}")

In [ ]:
def compute_wpe_direction(wpe_dir, output_path):
    """
    For each patient, calculate whether the WPE on average RISES or FALLS in pre-ictal vs inter-ictal epochs.
    Output: direction[p_id] = +1 (WPE rises pre-ictal) or -1 (WPE falls)
    """
    directions = {}
    files = sorted([f for f in os.listdir(wpe_dir) if f.endswith('_X_labeled.npy')])
    print("Calculating pre-ictal WPE direction by patient")
    for f in files:
        pid_str = f.replace('_X_labeled.npy', '')
        pid_int = int(''.join(filter(str.isdigit, pid_str)))
        X_p = np.load(os.path.join(wpe_dir, f)).astype(np.float32)
        y_p = np.load(os.path.join(wpe_dir, f"{pid_str}_y_labeled.npy"))
        mean_preictal    = X_p[y_p == 1].mean()
        mean_interictal  = X_p[y_p == 0].mean()
        diff             = mean_preictal - mean_interictal
        direction        = +1 if diff > 0 else -1
        directions[pid_int] = direction
        arrow = "Grows" if direction == +1 else "Falls"
        print(f"  chb{pid_int:02d}: pre-ictal WPE {arrow} (diff={diff:+.4f})")

    np.save(os.path.join(output_path, "wpe_directions.npy"), directions)
    print(f"Directions saved in: {output_path}/wpe_directions.npy")
    return directions
directions = compute_wpe_direction(CONFIG['WPE_DIR'], db_final_std_v2)

In [ ]:
#Execution
if __name__ == "__main__":
    patient_files = os.listdir(CONFIG['WPE_DIR'])
    patient_ids = sorted(set([f.split('_X_18ch')[0] for f in patient_files if '_X_18ch.npy' in f]))
    for p_id in patient_ids:
        label_patient(p_id)
    create_robust_standardized_dataset(CONFIG['WPE_DIR'], db_final_std_v2)